# DiGToR on FMB - end-to-end

Disagreement-Guided Token Routing for RGB-Infrared semantic segmentation.
Run the cells top to bottom on Colab (Pro / A100) or Kaggle; the notebook is
self-contained:

1. Clone the repo.
2. Download the FMB dataset from Google Drive and unzip `train.zip` / `test.zip`.
3. Train the v_only / t_only teachers, the fusion baseline, and DiGToR (80 epochs each).
4. Evaluate on the held-out test split: segmentation mIoU, the thermal rescue
   protocol, routing analyses, corruption robustness, FLOPs / latency, and the
   modality-cut check.

All results are written to `results_fmb/*.json`.

In [ ]:
import os
import shutil

repo_name = "DiGToR"

# Nếu thư mục đã tồn tại, xóa đi để chuẩn bị tải mới
if os.path.exists(repo_name):
    print(f"Phát hiện thư mục '{repo_name}' đã tồn tại. Đang xóa...")
    shutil.rmtree(repo_name)
    print("Đã xóa thư mục cũ.")

repo_url = f"https://github.com/nguyenmaiductrong/{repo_name}.git"

print(f"Đang tải repo từ {repo_url}...")
exit_code = os.system(f"git clone {repo_url}")

if exit_code == 0:
    print("Tải thành công. Kiểm tra ở thư mục Output/Working.")
else:
    print("Có lỗi xảy ra khi tải, kiểm tra lại đường dẫn mạng hoặc URL.")

In [ ]:
# repo da duoc clone moi o cell 0 (fresh = latest main); pull nay chi de chac chan
!cd DiGToR && git pull origin main || echo "(pull skipped - fresh clone already latest)"

In [ ]:
cd DiGToR

In [ ]:
# --- Download FMB data from Google Drive and unzip train/test ---
# Data is stored OUTSIDE the cloned repo so a re-clone does not wipe it, and the
# step is idempotent: re-running skips the ~1GB download if the data is ready.
import os, glob, zipfile, shutil, subprocess, sys

DRIVE_URL = "https://drive.google.com/drive/folders/1T_jVi80tjgyHTQDpn-TjfySyW4CK1LlF"
FMB_DIR = "/content/FMB" if os.path.isdir("/content") else os.path.join(
    os.path.dirname(os.getcwd()), "FMB")
_mods = ("Visible", "Infrared", "Label")

def _ready(split):
    return all(os.path.isdir(os.path.join(FMB_DIR, split, m)) for m in _mods)

if _ready("train") and _ready("test"):
    print("FMB already prepared at", FMB_DIR)
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
    import gdown
    dl = os.path.join(os.getcwd(), "_fmb_download")
    shutil.rmtree(dl, ignore_errors=True)
    os.makedirs(dl, exist_ok=True)
    gdown.download_folder(DRIVE_URL, output=dl, quiet=False, use_cookies=False)

    def _find(name):
        hits = glob.glob(os.path.join(dl, "**", name), recursive=True)
        if not hits:
            raise FileNotFoundError(f"{name} not found in the Drive folder")
        return hits[0]

    def _modroot(base):
        # the directory that directly holds Visible/Infrared/Label, whatever the
        # zip's internal nesting is
        for root, _, _ in os.walk(base):
            if all(os.path.isdir(os.path.join(root, m)) for m in _mods):
                return root
        raise FileNotFoundError(f"no Visible/Infrared/Label folder under {base}")

    os.makedirs(FMB_DIR, exist_ok=True)
    for split in ("train", "test"):
        tmp = os.path.join(os.getcwd(), f"_unzip_{split}")
        shutil.rmtree(tmp, ignore_errors=True)
        print(f"unzipping {split}.zip ...")
        with zipfile.ZipFile(_find(f"{split}.zip")) as z:
            z.extractall(tmp)
        dst = os.path.join(FMB_DIR, split)
        shutil.rmtree(dst, ignore_errors=True)
        shutil.move(_modroot(tmp), dst)
        shutil.rmtree(tmp, ignore_errors=True)
    shutil.rmtree(dl, ignore_errors=True)
    print("Prepared FMB at", FMB_DIR)

# The data-detection cell below reads FMB_ROOT first, so this works regardless of
# where the data was staged.
os.environ["FMB_ROOT"] = FMB_DIR
for split in ("train", "test"):
    n = len(glob.glob(os.path.join(FMB_DIR, split, "Label", "*"))) if _ready(split) else 0
    print(f"  {split}: {n} label files")

In [ ]:
# --- Repo + FMB data (auto-detect official train/test split OR legacy flat) ---
# Runs on Colab (Pro / A100) or Kaggle. The previous cell downloads the data and
# sets FMB_ROOT, which is checked first below; the other paths are fallbacks.
import os, sys, glob

REPO = os.getcwd()                 # current dir = the DiGToR repo
assert os.path.isfile(os.path.join(REPO, 'digtor', '__init__.py')),     f'Run this from the DiGToR repo root (no digtor/ package found in {REPO}).'

_mods = ['Visible', 'Infrared', 'Label']

def _is_fmb_root(path):
    flat = all(os.path.isdir(os.path.join(path, s)) for s in _mods)
    split = all(os.path.isdir(os.path.join(path, sp, s)) for sp in ['train', 'test'] for s in _mods)
    return flat or split, split

# Priority: FMB_ROOT (set by the download cell) -> repo/FMB -> Colab /content or
# Drive -> Kaggle inputs.
_candidates = []
if os.environ.get('FMB_ROOT'):
    _candidates.append(os.environ['FMB_ROOT'])
_candidates.append(os.path.join(REPO, 'FMB'))
_candidates += ['/content/FMB', '/content/drive/MyDrive/FMB']
_candidates += glob.glob('/kaggle/input/**/FMB', recursive=True)
_candidates += glob.glob('/kaggle/input/*', recursive=False)

ROOT = None
SPLIT_LAYOUT = False
for _cand in dict.fromkeys(_candidates):
    ok, split = _is_fmb_root(_cand)
    if ok:
        ROOT, SPLIT_LAYOUT = _cand, split
        break
assert ROOT is not None, 'FMB data not found: need flat Visible/Infrared/Label OR official train/test layout.'

print('FMB layout =', 'official train/test (held-out test)' if SPLIT_LAYOUT else 'legacy flat (auto-split)')
print('REPO =', REPO)
print('ROOT =', ROOT)

sys.path.insert(0, REPO)
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# --- Hyper-parameters: Phase-0 hard-seg-loss forced paths ---
# Pinned to the successful Phase-0 UNet rig: a 3-path DiGToR router
# (V-trust / T-rescue / Joint), rel_gate OFF at eval, hard seg_loss on
# force_path='v'/'t' plus teacher KD (--lambda_distill 0.5).
# The digtor.dataset.fmb loader uses the no-leakage data contract:
# FMB/train -> train+val only, FMB/test -> held-out evaluation.
import os
from digtor.models import build_model

# Guardrail: if this fails, the Kaggle clone is not on the expected 3-path code.
_probe = build_model('digtor', base=8)
assert getattr(_probe.router[-1], 'out_channels', None) == 3, 'Expected the 3-path DiGToR router.'
del _probe

H, W, BASE = 384, 512, 32
BS = 8
EPOCHS = 80
DIGTOR_EPOCHS = 80
CKPT, RES = 'ckpt_fmb', 'results_fmb'
AMP = '--amp'                      # set '' to disable mixed precision
LR = '--lr 5e-4'
IGNORE_BG = '--ignore_bg'          # match the G1/reporting convention; set '' to include class 0
CORRUPT = '--corrupt_aug --corrupt_p 0.5'
DISTILL = '--lambda_distill 0.5'   # hard seg-loss forced paths + KD when teachers exist
NOGATE = '--disable_gate'          # train/eval with reliability gate OFF (rel_gate=0)
GAMMA_PRIOR = '--gamma_prior 2.0'
LAMBDA_COST = '--lambda_cost 0.1'
ROUTE_BETA = '--route_beta 0.7'

# DRY RUN: set LIMIT='8' to smoke-test all cells quickly; set '' for full train/test.
LIMIT = ''
LIMIT_ARG = f'--limit {LIMIT}' if LIMIT else ''
if LIMIT:
    os.environ['FMB_LIMIT'] = LIMIT
    EPOCHS = DIGTOR_EPOCHS = 1
else:
    os.environ.pop('FMB_LIMIT', None)

os.makedirs(CKPT, exist_ok=True)
os.makedirs(RES, exist_ok=True)
print('config:', dict(H=H, W=W, BASE=BASE, BS=BS, EPOCHS=EPOCHS,
                      DIGTOR_EPOCHS=DIGTOR_EPOCHS, CKPT=CKPT, RES=RES,
                      LIMIT=LIMIT or 'full', lr=LR,
                      ignore_bg=bool(IGNORE_BG), corrupt_aug=CORRUPT,
                      distill=DISTILL, gate='off', route_beta=ROUTE_BETA))

In [ ]:
# --- Weights & Biases: checkpoint sync (survive Colab disconnects) ---
# Logs metrics and uploads each best checkpoint as a wandb artifact (<mode>-ckpt).
# A dropped session can pull them back (next cell) so finished models are reused
# instead of retrained. Set USE_WANDB=False to disable everything.
import os, subprocess, sys

USE_WANDB = True
WB_PROJECT = 'digtor-fmb'
WB_ENTITY = None            # None = your default wandb entity

if USE_WANDB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'wandb'], check=True)
    import wandb
    # Kaggle: add your key as a secret named WANDB_API_KEY. Colab: this prompts
    # once, or set os.environ['WANDB_API_KEY'] = '...' before running.
    try:
        wandb.login()
    except Exception as e:
        print('wandb.login failed -> disabling wandb:', e); USE_WANDB = False

WANDB = (f'--wandb --wandb_project {WB_PROJECT}'
         + (f' --wandb_entity {WB_ENTITY}' if WB_ENTITY else '')) if USE_WANDB else ''
ENTITY_ARG = f'--entity {WB_ENTITY}' if WB_ENTITY else ''
print('wandb:', 'ON' if USE_WANDB else 'OFF', '| WANDB =', repr(WANDB))


In [ ]:
# --- Pull any already-trained checkpoints from wandb into CKPT ---
# Idempotent + failsafe: missing artifacts are skipped. After this, each training
# cell's `[ -f {CKPT}/x.pt ]` guard reuses whatever was recovered instead of
# retraining. Run this first on a fresh runtime to resume a dropped session.
if USE_WANDB:
    !python -m digtor.wandb_ckpt --dataset fmb --pull --project {WB_PROJECT} {ENTITY_ARG} --out {CKPT} --modes v_only t_only fusion digtor
else:
    print('wandb OFF -> skipping checkpoint pull')


In [ ]:
# --- A100 speed knobs (quality-neutral) ---
# channels_last + persistent dataloader workers are always on in the code. Here
# we add torch.compile (fuses the conv graph -> faster GPU step) and match the
# worker count to the host CPU cores so the GPU is never starved waiting on JPEG
# decode. None of this changes the maths, so results are preserved.
import os as _os
WORKERS = f"--workers {min(8, (_os.cpu_count() or 4))}"
COMPILE = '--compile'        # set '' to skip torch.compile (e.g. if it errors)
SPEED = f'{COMPILE} {WORKERS}'.strip()
#
# OPTIONAL, NOT quality-neutral: a bigger batch fills the A100 better but changes
# the optimisation. If you raise BS, scale LR by the same factor (linear scaling
# rule) to keep accuracy, e.g. BS=16 -> LR='--lr 1e-3'. Left at the pinned BS=8.
# BS = 16; LR = '--lr 1e-3'
print('SPEED =', repr(SPEED))


In [ ]:
# --- Optional: stage matching Phase-0 checkpoints from an attached Kaggle Dataset ---
# Default is OFF because this notebook reruns training on the new split.
# Turn on only when the attached checkpoint dataset was trained with this same
# Phase-0 code and the same train/test split.
import glob, os, shutil

USE_INPUT_CKPTS = False
os.makedirs(CKPT, exist_ok=True)

def _stage(name):
    dst = os.path.join(CKPT, name)
    if os.path.exists(dst):
        print(f'[stage] {name}: already in {CKPT}/')
        return
    hits = [h for h in glob.glob(f'/kaggle/input/**/{name}', recursive=True)
            if 'mit' not in h.lower() and 'detector2' not in h.lower()]
    if hits:
        shutil.copy(hits[0], dst)
        print(f'[stage] {name} <- {hits[0]}')
    else:
        print(f'[stage] {name}: not found -> will train')

if USE_INPUT_CKPTS:
    for _n in ['v_only.pt', 't_only.pt', 'fusion.pt', 'digtor.pt']:
        _stage(_n)
else:
    print('[stage] checkpoint staging disabled; training/eval will use local outputs only')

In [ ]:
# Step 1 - V-only teacher (clean reference, train/val from FMB/train only)
![ -f {CKPT}/v_only.pt ] && echo "[skip] {CKPT}/v_only.pt exists - reusing" || python -m digtor.train --dataset fmb --root {ROOT} --mode v_only --epochs {EPOCHS} --bs {BS} --height {H} --width {W} --base {BASE} --out {CKPT} {AMP} {LR} {IGNORE_BG} {WANDB} {SPEED}


In [ ]:
# Step 2 - T-only teacher (clean reference, train/val from FMB/train only)
![ -f {CKPT}/t_only.pt ] && echo "[skip] {CKPT}/t_only.pt exists - reusing" || python -m digtor.train --dataset fmb --root {ROOT} --mode t_only --epochs {EPOCHS} --bs {BS} --height {H} --width {W} --base {BASE} --out {CKPT} {AMP} {LR} {IGNORE_BG} {WANDB} {SPEED}


In [ ]:
# Step 3 - rescue detector (decision gate) on the held-out test split
!python -m digtor.eval_detector --dataset fmb --root {ROOT} --height {H} --width {W} --base {BASE} --v_ckpt {CKPT}/v_only.pt --t_ckpt {CKPT}/t_only.pt --split test --out {RES}/detector.json {IGNORE_BG}

In [ ]:
# Step 4 - Fusion baseline (same corruption augmentation for a fair MFR comparison)
![ -f {CKPT}/fusion.pt ] && echo "[skip] {CKPT}/fusion.pt exists - reusing" || python -m digtor.train --dataset fmb --root {ROOT} --mode fusion --epochs {EPOCHS} --bs {BS} --height {H} --width {W} --base {BASE} --out {CKPT} {AMP} {LR} {IGNORE_BG} {WANDB} {SPEED} {CORRUPT}


### Step 5 - DiGToR Phase 0

The restored configuration: a 3-path DiGToR router with V-trust, T-rescue, and Joint paths. The key fix is training the forced pure paths with hard segmentation loss (`force_path='v'/'t'`) plus teacher KD, so catastrophic single-modality failure can cut to a strong fallback path.

In [ ]:
# Step 5 - DiGToR retrain with hard-seg-loss forced paths
![ -f {CKPT}/v_only.pt ] && [ -f {CKPT}/t_only.pt ] || echo 'WARN: missing teachers -> run Step 1/2 first'
![ -f {CKPT}/digtor.pt ] && echo "[skip] {CKPT}/digtor.pt exists - reusing" || python -m digtor.train --dataset fmb --root {ROOT} --mode digtor --epochs {DIGTOR_EPOCHS} --bs {BS} --height {H} --width {W} --base {BASE} --out {CKPT} --v_ckpt {CKPT}/v_only.pt --t_ckpt {CKPT}/t_only.pt {AMP} {LR} {IGNORE_BG} {WANDB} {SPEED} {CORRUPT} {GAMMA_PRIOR} {LAMBDA_COST} {ROUTE_BETA} {DISTILL} {NOGATE}

# Clean mIoU + rescue protocol + routing analyses on held-out FMB/test. rel_gate=0 at eval.
!python -m digtor.eval_rescue --dataset fmb --root {ROOT} --height {H} --width {W} --base {BASE} --ckpt_dir {CKPT} --rel_gate 0 --out {RES}/eval_rescue.json {IGNORE_BG}

In [ ]:
# Step 6 - corruption robustness, FLOPs, and the modality-cut win check
# Dense-routing robustness table (no retrain).
!python -m digtor.eval_robustness --dataset fmb --root {ROOT} --height {H} --width {W} --base {BASE} --ckpt_dir {CKPT} --rel_gate 0 --out {RES}/robustness.json {IGNORE_BG} {LIMIT_ARG}

# Route-share / FLOP accounting for the 3-path model.
!python -m digtor.profile_flops --dataset fmb --root {ROOT} --ckpt {CKPT}/digtor.pt --base {BASE} --height {H} --width {W} --out {RES}/flops.json {LIMIT_ARG}

# Catastrophic-thermal check with realizable force_path cuts.
!python -m digtor.eval_modality_cut --dataset fmb --root {ROOT} --height {H} --width {W} --base {BASE} --ckpt_dir {CKPT} --skips 0.3 0.5 0.6 --out {RES}/modality_cut.json {IGNORE_BG} {LIMIT_ARG}

In [ ]:
# Step 7 - Optional operating-point sweep
# The detector2d threshold sweep belongs to the later 2-path branch and is not
# part of this Phase-0 run. The comparable routing evidence is in
# eval_rescue.json, robustness.json, and modality_cut.json.
print('Skipped: threshold sweep is not part of this Phase-0 run.')

In [ ]:
# Step 8 - Phase-0 result summary
import json, math, os

def _ld(path):
    try:
        return json.load(open(path))
    except Exception as e:
        print(f'[missing] {path}: {e}')
        return None

er = _ld(f'{RES}/eval_rescue.json')
det = _ld(f'{RES}/detector.json')
rob = _ld(f'{RES}/robustness.json')
flp = _ld(f'{RES}/flops.json')
cut = _ld(f'{RES}/modality_cut.json')

def row(name, evidence, verdict=''):
    print(f'{name:30s} | {evidence:70s} | {verdict}')

print('=' * 128)
print('DiGToR Phase-0 summary: hard-seg-loss forced paths, new FMB split')
print('=' * 128)

if er and 'digtor' in er and 'fusion' in er:
    dm, fm = er['digtor']['mIoU'], er['fusion']['mIoU']
    row('clean mIoU', f'digtor {dm:.4f} vs fusion {fm:.4f} (delta {dm-fm:+.4f})',
        'WIN' if dm >= fm else 'below fusion')
if er and er.get('digtor_rescue') and er.get('fusion_rescue'):
    dr, fr = er['digtor_rescue'], er['fusion_rescue']
    row('rescue protocol',
        f"TRR {dr['TRR']:.3f}/{fr['TRR']:.3f}, VPR {dr['VPR']:.3f}/{fr['VPR']:.3f}, HRR {dr['HRR']:.3f}/{fr['HRR']:.3f}",
        'compare vs fusion')
if er and er.get('routing_alignment_pct'):
    ra = er['routing_alignment_pct']
    if 't_rescue' in ra and 'easy' in ra:
        tr_t = ra['t_rescue'][1]
        ez_t = ra['easy'][1]
        row('route selectivity', f't_rescue {tr_t:.1f}%T vs easy {ez_t:.1f}%T (spread {tr_t-ez_t:+.1f})')
if rob and rob.get('MFR'):
    m = rob['MFR']
    d = m.get('digtor', {}).get('thermal_MFR', float('nan'))
    f = m.get('fusion', {}).get('thermal_MFR', float('nan'))
    row('thermal MFR', f'digtor {d:.3f} vs fusion {f:.3f}', 'WIN' if d >= f else 'below fusion')
if cut and cut.get('verdict'):
    row('modality-cut', str(cut['verdict'])[:70])
if flp and flp.get('flops_gflops'):
    fg = flp['flops_gflops']
    if 'digtor' in fg and 'fusion' in fg:
        row('FLOPs', f"digtor {fg['digtor']:.1f} vs fusion {fg['fusion']:.1f} GFLOPs ({fg['digtor']/fg['fusion']:.2f}x)")
print('=' * 128)
print('Target: digtor should track fusion on clean, win most corruptions, and pass the cut check.')

In [ ]:
# Collect all JSON results for download (Output tab)
import json, glob
for f in sorted(glob.glob(f'{RES}/*.json')):
    print('='*60, '
', f)
    print(json.dumps(json.load(open(f)), indent=2)[:1500])
